State Distribution by Participant Profile: This notebook analyses the distribution of FSM states across the participant profiles.

Imports and Paths

In [ ]:
import pandas as pd

# Load shared project paths and configuration
%run 00_config.ipynb

Load FSM results

In [ ]:
INPUT_FILE = OUTPUT_DIR / "fsm_simulation_all_users.xlsx"

fsm_results = pd.read_excel(INPUT_FILE)

print(f"Total records: {len(fsm_results)}")

fsm_results.head()

Create analysis dataset: The FSM results are separated into a complete state dataset and a transition dataset excluding onboarding initialization records.

In [ ]:
state_df = fsm_results.copy()

transition_df = fsm_results[
    fsm_results["previous_state"] != "None (Onboarding Init)"
].copy()

print("State records:", len(state_df))
print("Transition records:", len(transition_df))

Load participants profile

In [ ]:
DAILY_DATASET_FILE = OUTPUT_DIR / "daily_dataset.xlsx"

daily_df = pd.read_excel(DAILY_DATASET_FILE)

In [ ]:
# Keep unique BMI profile information for each participant
profiles = (
    daily_df[["user_id", "bmi_profile"]]
    .drop_duplicates()
)

print(profiles["bmi_profile"].value_counts())

In [ ]:
# Add BMI profile information to FSM state records
state_df = state_df.merge(
    profiles,
    on="user_id",
    how="left"
)

State Distribution by Profile: number and percentage of state assignments within each participant profile.

In [ ]:
profile_state_distribution = (
    state_df
    .groupby(["bmi_profile", "new_state"])
    .size()
    .reset_index(name="n_days")
)

profile_state_distribution["percentage"] = (
    profile_state_distribution
    .groupby("bmi_profile")["n_days"]
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

profile_state_distribution = profile_state_distribution.sort_values(
    ["bmi_profile", "percentage"],
    ascending=[True, False]
)

profile_state_distribution

In [ ]:
profile_state_distribution.to_excel(
    RESULTS_DIR / "10_profile_state_distribution.xlsx",
    index=False
)

In [ ]:
profile_state_pivot = (
    profile_state_distribution
    .pivot(
        index="new_state",
        columns="bmi_profile",
        values="percentage"
    )
    .fillna(0)
)

profile_state_pivot